# Storm maps: metamodel and dynamic runs against the buoys

The largest storms of the dynamic year (08), with every estimate side by side:

- the **metamodel** (linear sum of the goal metamodels, bulk and spectral);
- the **dynamic runs** of `dynamic_runs`: the SnapWave run of 08 and, optionally, others
  over some of the same hours (another SnapWave build, SWAN).

Sections: 3 scores them against the **buoys** through each storm (the reference here), 4 maps
them at the storm hour, 5 compares the dynamic runs with one of them (`reference_run`, e.g.
SWAN) on the hours and sites they share.

Storms are the peaks of the goal-mean offshore Hs, at least `min_gap` apart, kept when the
first dynamic run saved a map (`save_map`) within `max_lag` of the peak. SnapWave clips the
boundary spread at 55° (notebook 08), the reconstruction does not: in broad-spread hours the
buoys are the referee. `tm01` only exists for the buoys and the metamodel (a stat of the
summed spectrum).

**Prerequisites:** dynamic runs postprocessed (08, `Dynamic_Cases_<period>.nc`), fitted models
(06), buoy files (03). The reconstructions are read from the cache (`tmp/reconstruct/`, filled
by 09 or a script); anything missing is computed here.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
from utils import metamodel as mm
from utils import simulation as sim
from utils import validation as val
from utils import viz
from utils.config import get_offshore_wave, open_offshore_waves, resolve

## Configuration

In [ ]:
period = "2013"  # a year of the dynamic run
offshore_name = "WHACS_cal"
wind = False  # dynamic run with or without wind (08); the metamodel has none
mda_n = 600  # must match a fitted mda_candidates entry
map_set = "isobaths"  # metamodel set on the maps
buoy_set = "buoys"  # metamodel set at the buoys
wl = "forcing"  # same water level as the dynamic run
n_storms = 4
min_gap = "4D"  # least time between two storm peaks
max_lag = "6h"  # saved map at most this far from the peak
window = pd.Timedelta("3D")  # half-width of the buoy window
extent = None  # (lon_min, lon_max, lat_min, lat_max) to zoom the maps; None = the mesh
series_variables = ["hs", "tp", "tm01", "dir"]  # buoy time series
compare_variables = ["hs", "tp", "dir"]  # metrics tables

cases = sim.cases_dir("dynamic", offshore=offshore_name, wind=wind)
# Dynamic runs: label -> Dynamic_Cases file (its maps in the case folders next to it). The first
# is the run of 08 and picks the storms; the others may cover only some hours. Missing files
# are skipped.
dynamic_runs = {"SnapWave": cases / f"Dynamic_Cases_{period}.nc"}
reference_run = None  # label of dynamic_runs the others are compared with in section 5
print(f"dynamic runs: {dynamic_runs}")

## 1 · Storms with a saved map

In [ ]:
first = next(iter(dynamic_runs))
offshore = open_offshore_waves(entry=get_offshore_wave(offshore_name)).sel(time=period).load()
storms = val.storm_peaks(offshore, n_storms, min_gap=min_gap, max_lag=max_lag,
                         map_hours=sim.saved_map_hours(Path(dynamic_runs[first]).parent))
storms.round(2)

## 2 · Metamodel, dynamic runs and buoys

In [ ]:
# Resources for the reconstructions (at most 75% of this machine). Edit res if wanted.
res = mm.reconstruction_resources(
    period, mda_n, metamodel=map_set, spectral=True,
    offshore_name=offshore_name, save_spectra=False, wl=wl,
)

In [ ]:
paths = {label: Path(path) for label, path in dynamic_runs.items() if Path(path).is_file()}
primary = xr.open_dataset(paths[first]).sel(time=period).load()
runs = {label: val.on_sites_of(xr.open_dataset(path).load(), primary) for label, path in paths.items()}
run_maps = {label: sim.saved_map_hours(path.parent) for label, path in paths.items()}


def metamodel(spectral: bool) -> xr.Dataset:
    parts = [mm.reconstruct(period, mda_n, metamodel=s, spectral=spectral, offshore_name=offshore_name,
                            resources=res, save_spectra=False, wl=wl) for s in (map_set, buoy_set)]
    both = xr.concat(parts, "sites", coords="minimal", compat="override")
    return val.on_sites_of(both.isel(sites=~both.get_index("sites").duplicated()), primary)


estimates = {"Metamodel bulk": metamodel(False), "Metamodel spectral": metamodel(True)} | runs
buoy = val.on_sites_of(val.load_buoy_observations(period, variables=series_variables), primary)
for label, est in estimates.items():
    print(f"{label:20s} {est.sizes['time']:5d} hours, {est.sizes['sites']:5d} sites, "
          f"{len(run_maps.get(label, []))} maps")
print(f"buoys with data: {list(buoy.sites.values)}; missing runs: {sorted(set(dynamic_runs) - set(paths))}")

## 3 · Against the buoys

Per storm, from `window` before to `window` after the peak, at the buoys with data: bias and
RMSE of every estimate covering the storm (one row each, the variables side by side), on the
hours they all have (a run of only some hours narrows them); the Hs peak of each (value, time,
minus the buoy, lag); and the time series.

In [ ]:
for storm in storms.itertuples():
    span = slice(storm.peak - window, storm.peak + window)
    obs = buoy.sel(time=span)
    sites = [s for s in obs.sites.values if int(obs["hs"].sel(sites=s).notnull().sum())]
    if not sites:
        print(f"{storm.peak:%Y-%m-%d}: no buoy data")
        continue
    obs = obs.sel(sites=sites)
    present = {label: est.sel(sites=sites).reindex(time=obs.time) for label, est in estimates.items()
               if est.sel(time=span).sizes["time"]}
    print(f"Storm {storm.peak:%Y-%m-%d %H:%M}, against the buoys: {', '.join(present)}")
    metrics = val.common_hours_metrics(obs, present, compare_variables)
    print(f"hours every estimate has, per buoy: {metrics.groupby('site')['n'].max().to_dict()}")
    table = metrics.set_index(["site", "estimate", "variable"])[["bias", "rmse"]].unstack("variable")
    columns = [(m, v) for v in compare_variables for m in ("bias", "rmse") if (m, v) in table]
    display(table[columns].swaplevel(axis=1).reindex(list(present), level="estimate").round(2))
    display(val.event_peaks(obs, present).round(2))
    for site in sites:
        val.plot_buoy_timeseries(obs, present, site, variables=series_variables)
        plt.suptitle(f"{site} - storm {storm.peak:%Y-%m-%d}")
        plt.show()

## 4 · Maps of the storm hour

Hs, then mean direction, of every estimate at the storm's map hour on one colour scale, the
observed buoy values on every panel. Hs: the dynamic runs on the mesh where their map was
saved (else at the sites). Direction: every estimate at the sites (SnapWave maps hold none).
The metamodel at the sites of `map_set` and `buoy_set`.

In [ ]:
tri = viz.storm_triangulation(paths[first].parent / f"{storms['map_hour'].iloc[0]:%Y%m%d %H%M%S}")
for storm in storms.itertuples():
    t = storm.map_hour
    maps = {label: sim.load_storm_map(paths[label].parent / f"{t:%Y%m%d %H%M%S}")
            for label in runs if t in run_maps[label]}
    obs = buoy.sel(time=t) if t in buoy.get_index("time") else None
    for variable in ("hs", "dir"):
        fields = {}
        for label, est in estimates.items():
            if label in maps and variable == "hs":  # direction at the sites for all (SnapWave maps have none)
                fields[label] = maps[label][variable]
            elif t in est.get_index("time") and bool(est[variable].sel(time=t).notnull().any()):
                fields[label] = est[variable].sel(time=t)
        observed = obs[variable].dropna("sites") if obs is not None and variable in obs else None
        viz.plot_storm_fields(fields, tri, variable, observed=observed, extent=extent,
                              title=f"{variable}, storm peak {storm.peak:%Y-%m-%d %H:%M} (map {t:%H:%M}), "
                                    f"goal-mean offshore Hs {storm.goal_mean_hs:.1f} m")
        plt.show()

## 5 · Dynamic runs against `reference_run`

The other dynamic runs against the reference (e.g. SnapWave builds against SWAN) on the hours
and sites they share, per site group and variable; then, at the storm map hours the reference
saved, the difference maps: Hs node by node on the mesh, direction at the sites (SnapWave maps
hold no direction). Red: higher than the reference (or turned clockwise).

In [ ]:
if reference_run in runs:
    ref = runs[reference_run]
    rows = [val.compare(ref, run, variables=[v for v in compare_variables if v in run]).metrics.assign(run=label)
            for label, run in runs.items() if label != reference_run]
    for variable, table in pd.concat(rows).groupby("variable", sort=False):
        print(f"{variable}: run - {reference_run}, {ref.sizes['time']} hours")
        display(table.pivot_table(index="group", columns="run", values=["bias", "rmse", "r2"]).round(3))
else:
    print(f"reference run {reference_run!r} not among {list(runs)}")

In [ ]:
for storm in storms.itertuples() if reference_run in runs else []:
    t = storm.map_hour
    if t not in run_maps[reference_run]:
        continue
    others = [label for label in runs if label != reference_run and t in run_maps[label]]
    ref_map = sim.load_storm_map(paths[reference_run].parent / f"{t:%Y%m%d %H%M%S}")
    hs = {label: sim.load_storm_map(paths[label].parent / f"{t:%Y%m%d %H%M%S}")["hs"] for label in others}
    viz.plot_storm_fields(hs, tri, "hs", reference=ref_map["hs"], reference_label=reference_run,
                          extent=extent, ncols=len(hs), title=f"Hs against {reference_run}, {t:%Y-%m-%d %H:%M}")
    plt.show()
    direction = {label: runs[label]["dir"].sel(time=t) for label in others}
    viz.plot_storm_fields(direction, tri, "dir", reference=runs[reference_run]["dir"].sel(time=t),
                          reference_label=reference_run, extent=extent, ncols=len(direction),
                          title=f"Direction against {reference_run}, {t:%Y-%m-%d %H:%M}")
    plt.show()